In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, recall_score, precision_score

# Додати корінь проєкту до шляхів імпорту, щоб бачити модуль src
if ".." not in sys.path:
    sys.path.append("..")

from src.dynamic_pipeline_builder import DynamicPipelineBuilder, validate_config

# 1. Зчитування файлу
csv_path = "../E_Commerce_Dataset.csv" if os.path.exists("../E_Commerce_Dataset.csv") else "E_Commerce_Dataset.csv"
df = pd.read_csv(csv_path, sep=None, engine="python")

# Очищення назви колонок від випадкових пробілів
df.columns = df.columns.str.strip()

print(f"Дані успішно завантажено! Розмірність: {df.shape}")
print("Фактичні колонки в датасеті:\n", df.columns.tolist())

Дані успішно завантажено! Розмірність: (3333, 11)
Фактичні колонки в датасеті:
 ['Churn', 'AccountWeeks', 'ContractRenewal', 'DataPlan', 'DataUsage', 'CustServCalls', 'DayMins', 'DayCalls', 'MonthlyCharge', 'OverageFee', 'RoamMins']


In [5]:
df["y_true"] = df["Churn"].astype(int)

# Евристичне бізнес-правило без ML:
# Клієнт схильний до відтоку, якщо він НЕ продовжив контракт (ContractRenewal == 0)
# АБО телефонував у підтримку більше 3 разів (CustServCalls >= 4)
rule = (df["ContractRenewal"] == 0) | (df["CustServCalls"] >= 4)
df["y_heuristic"] = rule.astype(int)

acc = accuracy_score(df["y_true"], df["y_heuristic"])
rec = recall_score(df["y_true"], df["y_heuristic"], pos_label=1)
prec = precision_score(df["y_true"], df["y_heuristic"], pos_label=1)

print("-" * 40)
print(f"Accuracy : {acc:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"Precision: {prec:.4f}")
print("-" * 40)

----------------------------------------
Accuracy : 0.8401
Recall   : 0.5300
Precision: 0.4555
----------------------------------------


In [ ]:
# 1. Демонстрація перехоплення витоку (навмисне внесення AccountWeeks у features)
bad_config = {
    "protected_attributes": ["AccountWeeks"],
    "features": {
        "numeric_continuous": ["AccountWeeks", "MonthlyCharge"]
    }
}

try:
    validate_config(bad_config)
    print("ПОМИЛКА: Валідатор не зупинив витік!")
except ValueError as e:
    print(f"Успіх: Валідатор перехопив витік захищеної ознаки:\n--> {e}\n")

# 2. Збірка конвеєра з оновленої конфігурації
config_path = "../config/config_lr.yaml" if os.path.exists("../config/config_lr.yaml") else "config/config_lr.yaml"
builder = DynamicPipelineBuilder(config_path)
pipeline = builder.build()

# Збірка списків колонок із конфігурації
feature_cols = []
for group in builder.config["features"].values():
    feature_cols.extend(group)

# Навчання трансформера на даних (лише препроцесинг)
pipeline.fit(df[feature_cols])

# Виведення аудиторського сліду
print("--- Аудиторський слід AdaptiveImputer (частки пропусків) ---")
for name, step, cols in pipeline.transformers_:
    if hasattr(step, "named_steps") and "imputer" in step.named_steps:
        audit = step.named_steps["imputer"].audit_trail_
        for idx, stats in audit.items():
            col_name = cols[idx] if isinstance(idx, int) and idx < len(cols) else idx
            print(f"Колонка: {col_name:<20} | Пропуски: {stats['missing_share']:.4f} | Стратегія: {stats['rule']['strategy']}")

Успіх: Валідатор перехопив витік захищеної ознаки:
--> Захищені ознаки потрапили у features: {'AccountWeeks'}

--- Аудиторський слід AdaptiveImputer (частки пропусків) ---
Колонка: DataUsage            | Пропуски: 0.0000 | Стратегія: median
Колонка: DayMins              | Пропуски: 0.0000 | Стратегія: median
Колонка: DayCalls             | Пропуски: 0.0000 | Стратегія: median
Колонка: MonthlyCharge        | Пропуски: 0.0000 | Стратегія: median
Колонка: OverageFee           | Пропуски: 0.0000 | Стратегія: median
Колонка: RoamMins             | Пропуски: 0.0000 | Стратегія: median
Колонка: DataPlan             | Пропуски: 0.0000 | Стратегія: most_frequent
Колонка: ContractRenewal      | Пропуски: 0.0000 | Стратегія: most_frequent
Колонка: CustServCalls        | Пропуски: 0.0000 | Стратегія: most_frequent
